# Batches And BatchManager Exploration

This notebook demonstrates the batching utilities in `liesel.optim.batch`. It focuses on the behavior of `Batches` and `BatchManager` without running a full optimization.

In [1]:
import warnings

import jax
import jax.numpy as jnp
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.model as lsl
import liesel.optim as opt
from liesel.optim.types import Position

## A Single Batches Object

`Batches` owns the observation indices for one sample size. If `n` is not divisible by `batch_size`, only complete mini-batches are used.

In [2]:
position = Position({"x": jnp.arange(10.0)})
batches = opt.Batches(["x"], n=10, batch_size=4, shuffle=False)

{
    "batches": batches,
    "batch_indices": batches.batch_indices.tolist(),
    "n_full_batches": batches.n_full_batches,
    "batch_share": batches.batch_share,
    "first_batch": batches.get_batched_position(position, 0)["x"].tolist(),
    "second_batch": batches.get_batched_position(position, 1)["x"].tolist(),
}

{'batches': Batches(n=10, batch_size=4, default_axis=0),
 'batch_indices': [[0, 1, 2, 3], [4, 5, 6, 7]],
 'n_full_batches': 2,
 'batch_share': 2.5,
 'first_batch': [0.0, 1.0, 2.0, 3.0],
 'second_batch': [4.0, 5.0, 6.0, 7.0]}

Passing `batch_size=None` disables mini-batching and creates one full-data batch.

In [3]:
full_data_batches = opt.Batches(["x"], n=10, batch_size=None, shuffle=False)

{
    "batch_size": full_data_batches.batch_size,
    "n_full_batches": full_data_batches.n_full_batches,
    "is_full_data": full_data_batches.is_full_data,
    "batch_indices": full_data_batches.batch_indices.tolist(),
}

{'batch_size': 10,
 'n_full_batches': 1,
 'is_full_data': True,
 'batch_indices': [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]}

## Epoch Shuffling

`start_epoch()` updates the stored observation order. With a fixed key, the shuffled order is reproducible.

In [4]:
shuffled_1 = opt.Batches(["x"], n=10, batch_size=4, shuffle=True).start_epoch(
    jax.random.key(3)
)
shuffled_2 = opt.Batches(["x"], n=10, batch_size=4, shuffle=True).start_epoch(
    jax.random.key(3)
)

{
    "indices": shuffled_1.indices.tolist(),
    "batch_indices": shuffled_1.batch_indices.tolist(),
    "same_key_same_indices": bool(jnp.all(shuffled_1.indices == shuffled_2.indices)),
}

{'indices': [0, 5, 6, 7, 4, 2, 1, 3, 9, 8],
 'batch_indices': [[0, 5, 6, 7], [4, 2, 1, 3]],
 'same_key_same_indices': True}

## Batching Along Different Axes

Each position key can use its own batching axis. Keys not listed in `axes` use `default_axis`.

In [5]:
matrix_position = Position(
    {
        "x": jnp.arange(15).reshape(3, 5),
        "y": jnp.arange(20).reshape(5, 4),
    }
)

axis_batches = opt.Batches(
    ["x", "y"],
    n=5,
    batch_size=2,
    axes={"x": 1},
    default_axis=0,
    shuffle=False,
)
axis_batch = axis_batches.get_batched_position(matrix_position, 0)

{
    "x_shape": axis_batch["x"].shape,
    "y_shape": axis_batch["y"].shape,
    "x_batch": axis_batch["x"].tolist(),
    "y_batch": axis_batch["y"].tolist(),
}

{'x_shape': (3, 2),
 'y_shape': (2, 4),
 'x_batch': [[0, 1], [5, 6], [10, 11]],
 'y_batch': [[0, 1, 2, 3], [4, 5, 6, 7]]}

## Model Convenience For One Observation Size

`Batches.from_model()` can infer the observed keys and sample size for a regular single-size model.

In [6]:
z = lsl.Var.new_obs(jnp.linspace(-1.0, 1.0, 12), name="z")
single_size_model = lsl.Model([z])

model_batches = opt.Batches.from_model(
    single_size_model,
    batch_size=3,
    position_keys=["z"],
)

{
    "type": type(model_batches).__name__,
    "position_keys": model_batches.position_keys,
    "n": model_batches.n,
    "batch_size": model_batches.batch_size,
    "n_full_batches": model_batches.n_full_batches,
}

{'type': 'Batches',
 'position_keys': ['z'],
 'n': 12,
 'batch_size': 3,
 'n_full_batches': 4}

## Oversized Batches With Replacement

A single `Batches` object rejects `batch_size > n` by default. With `sample_with_replacement=True`, an oversized batch is filled by sampling valid observation rows.

In [7]:
oversized = opt.Batches(
    ["small"],
    n=5,
    batch_size=8,
    shuffle=True,
    sample_with_replacement=True,
).start_epoch(jax.random.key(1))

{
    "n_full_batches": oversized.n_full_batches,
    "batch_indices": oversized.batch_indices.tolist(),
    "all_indices_in_bounds": bool(jnp.all(oversized.batch_indices < oversized.n)),
}

{'n_full_batches': 1,
 'batch_indices': [[1, 2, 0, 3, 3, 4, 3, 1]],
 'all_indices_in_bounds': True}

## Strict BatchManager

`BatchManager` combines several `Batches` objects. In strict mode, all children must have the same number of full batches.

In [8]:
strict_manager = opt.BatchManager(
    [
        opt.Batches(["x"], n=6, batch_size=2, shuffle=False),
        opt.Batches(["y"], n=9, batch_size=3, shuffle=False),
    ],
    mode="strict",
)
strict_position = Position({"x": jnp.arange(6), "y": jnp.arange(9)})
strict_batch = strict_manager.get_batched_position(strict_position, 1)

{
    "position_keys": strict_manager.position_keys,
    "n": strict_manager.n,
    "batch_size": strict_manager.batch_size,
    "batch_shares": strict_manager.batch_shares,
    "n_full_batches": strict_manager.n_full_batches,
    "batch_x": strict_batch["x"].tolist(),
    "batch_y": strict_batch["y"].tolist(),
}

{'position_keys': ['x', 'y'],
 'n': (6, 9),
 'batch_size': (2, 3),
 'batch_shares': (3.0, 3.0),
 'n_full_batches': 3,
 'batch_x': [2, 3],
 'batch_y': [3, 4, 5]}

## Resampling BatchManager

In resample mode, children may have different numbers of full batches. The joint epoch length is controlled by `epoch_size`.

In [9]:
def make_resample_manager(epoch_size="max"):
    return opt.BatchManager(
        [
            opt.Batches(["x"], n=6, batch_size=2, shuffle=True),
            opt.Batches(["y"], n=8, batch_size=4, shuffle=True),
        ],
        mode="resample",
        epoch_size=epoch_size,
    )


resample_max = make_resample_manager("max").start_epoch(jax.random.key(4))
resample_min = make_resample_manager("min")
resample_manual = make_resample_manager(5).start_epoch(jax.random.key(4))

{
    "epoch_lengths": {
        "max": resample_max.n_full_batches,
        "min": resample_min.n_full_batches,
        "manual": resample_manual.n_full_batches,
    },
    "max_batch_numbers": resample_max.batch_numbers.tolist(),
    "manual_batch_numbers": resample_manual.batch_numbers.tolist(),
}

{'epoch_lengths': {'max': 3, 'min': 2, 'manual': 5},
 'max_batch_numbers': [[2, 1], [1, 0], [0, 1]],
 'manual_batch_numbers': [[2, 1], [1, 0], [0, 1], [1, 0], [0, 0]]}

Scalar `batch_share` is available only when every child has the same likelihood scale. Otherwise, use `scaled_log_lik()` for per-branch scaling.

In [10]:
equal_share = strict_manager.batch_share

unequal_manager = opt.BatchManager(
    [
        opt.Batches(["x"], n=10, batch_size=5),
        opt.Batches(["y"], n=9, batch_size=4),
    ]
)

try:
    unequal_manager.batch_share
except ValueError as error:
    unequal_share_error = str(error)

{
    "equal_share": equal_share,
    "unequal_batch_shares": unequal_manager.batch_shares,
    "unequal_share_error": unequal_share_error,
}

{'equal_share': 3.0,
 'unequal_batch_shares': (2.0, 2.25),
 'unequal_share_error': 'BatchManager.batch_share is only available when all contained Batches objects have the same n / batch_size. Use per-branch scaling via BatchManager.scaled_log_lik() instead.'}

## Automatic Grouping For Multiple Sizes

`BatchManager.from_model()` groups observed variables by inferred sample size. `Batches.from_model(..., multi_size="manager")` returns a manager only when multiple sizes are present.

In [11]:
x_obs = lsl.Var.new_obs(jnp.arange(12.0), name="x_obs")
y_obs = lsl.Var.new_obs(jnp.arange(5.0), name="y_obs")
multi_size_model = lsl.Model([x_obs, y_obs])

grouped_manager = opt.BatchManager.from_model(
    multi_size_model,
    batch_size=3,
    position_keys=["x_obs", "y_obs"],
)
managed_from_batches = opt.Batches.from_model(
    multi_size_model,
    batch_size=3,
    position_keys=["x_obs", "y_obs"],
    multi_size="manager",
)

{
    "grouped_type": type(grouped_manager).__name__,
    "from_batches_type": type(managed_from_batches).__name__,
    "n": grouped_manager.n,
    "batch_size": grouped_manager.batch_size,
    "n_full_batches": grouped_manager.n_full_batches,
    "child_position_keys": [child.position_keys for child in grouped_manager.batches],
}

{'grouped_type': 'BatchManager',
 'from_batches_type': 'BatchManager',
 'n': (12, 5),
 'batch_size': (3, 3),
 'n_full_batches': 4,
 'child_position_keys': [['x_obs'], ['y_obs']]}

Passing `batch_size=None` creates one full-data child batch per sample-size group.

In [12]:
full_data_manager = opt.BatchManager.from_model(
    multi_size_model,
    batch_size=None,
    position_keys=["x_obs", "y_obs"],
)

{
    "is_full_data": full_data_manager.is_full_data,
    "n": full_data_manager.n,
    "batch_size": full_data_manager.batch_size,
    "n_full_batches": full_data_manager.n_full_batches,
}

{'is_full_data': True,
 'n': (12, 5),
 'batch_size': (12, 5),
 'n_full_batches': 1}

## Common Batch Size With A Small Branch

In resample mode, `BatchManager.from_model()` can use a common `batch_size` that is larger than a smaller branch. The small branch then samples rows with replacement.

In [ ]:
common_size_manager = opt.BatchManager.from_model(
    multi_size_model,
    batch_size=6,
    position_keys=["x_obs", "y_obs"],
).start_epoch(jax.random.key(8))
common_size_batch = common_size_manager.get_batched_position(
    multi_size_model.extract_position(["x_obs", "y_obs"]),
    batch_index=0,
)

{
    "n": common_size_manager.n,
    "batch_size": common_size_manager.batch_size,
    "small_branch_uses_replacement": common_size_manager.batches[
        1
    ].sample_with_replacement,
    "batch_shapes": {key: value.shape for key, value in common_size_batch.items()},
    "small_branch_values": common_size_batch["y_obs"].tolist(),
}

{'n': (12, 5),
 'batch_size': (6, 6),
 'small_branch_uses_replacement': True,
 'batch_shapes': {'x_obs': (6,), 'y_obs': (6,)},
 'small_branch_values': [4.0, 0.0, 2.0, 1.0, 3.0, 1.0]}

## Per-Branch Likelihood Scaling

`BatchManager.scaled_log_lik()` scales each observed branch by its own `n / batch_size` factor.

In [14]:
y1 = lsl.Var.new_obs(
    jnp.arange(6.0),
    lsl.Dist(tfd.Normal, loc=0.0, scale=1.0),
    name="y1",
)
y2 = lsl.Var.new_obs(
    jnp.arange(8.0),
    lsl.Dist(tfd.Normal, loc=0.0, scale=1.0),
    name="y2",
)
lik_model = lsl.Model([y1, y2])

lik_manager = opt.BatchManager(
    [
        opt.Batches(["y1"], n=6, batch_size=2, shuffle=False),
        opt.Batches(["y2"], n=8, batch_size=4, shuffle=False),
    ],
    mode="resample",
    epoch_size="max",
)
lik_batch = lik_manager.get_batched_position(
    lik_model.extract_position(["y1", "y2"]),
    batch_index=0,
)
lik_state = lik_model.update_state(lik_batch, lik_model.state)

manual_scaled_log_lik = (
    lik_manager.batches[0].batch_share * lik_state[y1.dist_node.name].value.sum()
    + lik_manager.batches[1].batch_share * lik_state[y2.dist_node.name].value.sum()
)

{
    "batch_shares": lik_manager.batch_shares,
    "scaled_log_lik": lik_manager.scaled_log_lik(lik_model, lik_state),
    "manual_scaled_log_lik": manual_scaled_log_lik,
    "matches_manual": bool(
        jnp.allclose(
            lik_manager.scaled_log_lik(lik_model, lik_state),
            manual_scaled_log_lik,
        )
    ),
}

{'batch_shares': (3.0, 2.0),
 'scaled_log_lik': Array(-28.365139, dtype=float32),
 'manual_scaled_log_lik': Array(-28.365139, dtype=float32),
 'matches_manual': True}

## Helpful Errors And Warnings

The scalar constructor rejects multi-size observed data by default. Strict managers reject unequal child batch counts, and resampling without observation-level shuffling emits a warning.

In [15]:
messages = {}

try:
    opt.Batches.from_model(
        multi_size_model,
        batch_size=3,
        position_keys=["x_obs", "y_obs"],
    )
except ValueError as error:
    messages["multi_size_default"] = str(error)

try:
    opt.BatchManager.from_model(
        multi_size_model,
        batch_size=3,
        position_keys=["x_obs", "y_obs"],
        mode="strict",
    )
except ValueError as error:
    messages["strict_unequal_counts"] = str(error)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    opt.BatchManager.from_model(
        multi_size_model,
        batch_size=3,
        position_keys=["x_obs", "y_obs"],
        shuffle=False,
    )
    messages["resample_without_shuffle"] = str(caught[0].message)

messages

{'multi_size_default': "Batches.from_model() found observed variables with different sample sizes: {12: ['x_obs'], 5: ['y_obs']}. Use Batches.from_model(..., multi_size='manager') or BatchManager.from_model(...).",
 'strict_unequal_counts': "mode='strict' requires all contained Batches objects to have the same n_full_batches, but got [4, 1].",
 'resample_without_shuffle': "BatchManager.from_model(..., mode='resample', shuffle=False) resamples child batch rows but leaves observations within each child in deterministic order. Set shuffle=True for stochastic observation-level batches."}

Useful knobs to change while exploring: `batch_size`, `shuffle`, per-key `axes`, `mode`, `epoch_size`, `sample_with_replacement`, and whether to use `Batches.from_model(..., multi_size="manager")` or `BatchManager.from_model()` explicitly.